In [7]:
dic = {
'PASP001':'s15363',
'PASP002':'s15508',
'PASP004':'15763',
'PASP014':'17105',
'PASP017':'15913',
'PASP019':'16850',
'PASP020':'17053',
'PASP021':'16803',
'PASP022':'16190',
'PASP023':'16408',
'PASP028':'16591',
'PASP029':'16547',
'PASP030':'16527',
'PASP031':'16342',
'PASP033':'16883'
}


Patient_list = ['PASP001_md1_NL_AR', 'PASP002_lr_AR_NL','PASP003_kb_NR_AL', 'PASP004_fm_NR_AL', 'PASP005_je_AL_NR', 
                'PASP006_ed','PASP007_kj_AR_NL', 'PASP008_gv_NR_AL', 'PASP009_kk_AR_NL', 'PASP010_ka', 'PASP011_gc_NL_AR',
                'PASP012_ct_AR_NL', 'PASP013_yd_NL_AR', 'PASP014_bs_NL_AR','PASP015_da_NR_AL', 'PASP016_ca_NR_AL',
                'PASP017_md2_AL_NR', 'PASP018_ht','PASP019_bj2_AR_NL', 'PASP020_rp_NL_AR', 'PASP021_bj1_AL_NR',
                'PASP022_sa_NL_AR', 'PASP023_ie_NL_AR', 'PASP025_wl_AR_NL', 'PASP026_mj_AL_NR', 'PASP027_me_AL_NR',
                'PASP028_lt_NL_AR', 'PASP029_wr_AL_NR', 'PASP030_hj_NR_AL', 'PASP031_pp_AL_NR', 'PASP032_nd_AL_NR',
                'PASP033_sa2_NL_AR']

Patient_list = ['PASP003_kb_NR_AL']

Patient_list_with_MRI = ['PASP001_md1_NL_AR', 'PASP002_lr_AR_NL', 'PASP004_fm_NR_AL', 'PASP014_bs_NL_AR', 'PASP017_md2_AL_NR',
                            'PASP022_sa_NL_AR', 'PASP023_ie_NL_AR', 'PASP028_lt_NL_AR', 'PASP029_wr_AL_NR', 'PASP030_hj_NR_AL',
                            'PASP031_pp_AL_NR','PASP019_bj2_AR_NL' , 'PASP021_bj1_AL_NR', 'PASP020_rp_NL_AR', 'PASP033_sa2_NL_AR']


In [8]:
import numpy as np
import matplotlib.pyplot as plt

import mne
from mne.minimum_norm import make_inverse_operator, apply_inverse, source_band_induced_power, apply_inverse_epochs,compute_source_psd, compute_source_psd_epochs
from mne.datasets import fetch_fsaverage


In [10]:
from mne.datasets import sample
from mne.datasets import eegbci
from mne.datasets import fetch_fsaverage
import os.path as op
import os

src_num = 4

# ==============================
# Create forward solution folder ONCE
# ==============================
forward_sol_folder = "/mnt/isilon/w_neuro/gopalarlab/Pain_project/data/PASP_MEG_data/Forward_files"
os.makedirs(forward_sol_folder, exist_ok=True)



for sub in Patient_list:
    
    num = '2'
    if sub in ['PASP001_md1_NL_AR', 'PASP017_md2_AL_NR', 'PASP021_bj1_AL_NR', 'PASP019_bj2_AR_NL', 'PASP033_sa2_NL_AR']:

        val = sub[7:11]
    else:
        val = sub[7:10]
        
    spont = 'SPONT_' + num + val + '_raw_quat_tsss.fif'
    sub_name = sub[0:7]

    if sub in ["PASP003_kb_NR_AL", "PASP006_ed", "PASP010_ka", "PASP018_ht"]:
        path = f'/mnt/isilon/w_neuro/gopalarlab/PASP_MEG_data/Resting_State_only/{sub}'
    else:
        path = f'/mnt/isilon/w_neuro/gopalarlab/PASP_MEG_data/{sub}'
       
    file_name = path+'/'+spont
    raw = mne.io.read_raw_fif(file_name)
    info = mne.io.read_info(file_name)
    
    mri_data_path = "/mnt/isilon/w_neuro/gopalarlab/Pain_project/data/Freesurfer_Files"
    
    if sub in Patient_list_with_MRI:
        # The paths to Freesurfer reconstructions
        subjects_dir = mri_data_path+'/subject'+dic[sub_name]
        subject = "sample"
        trans = path+ "/audvis_raw-trans.fif"
        src = mne.setup_source_space(
            subject, spacing=f"ico{src_num}", add_dist="patch", subjects_dir=subjects_dir
        )

        conductivity = (0.3,)  # for single layer
        # conductivity = (0.3, 0.006, 0.3)  # for three layers
        model = mne.make_bem_model(
            subject=subject, ico=4, conductivity=conductivity, subjects_dir=subjects_dir
        )
        bem = mne.make_bem_solution(model)
    
    else:
        fs_dir = fetch_fsaverage(verbose=True)
        subjects_dir = op.dirname(fs_dir)
        
        # The files live in:
        subject = "fsaverage"
        trans = "fsaverage"  # MNE has a built-in fsaverage transformation
        bem = fs_dir / "bem" / "fsaverage-5120-5120-5120-bem-sol.fif"

        src = mne.setup_source_space(
            subject, spacing=f"ico{src_num}", add_dist="patch", subjects_dir=subjects_dir
        )

    fwd = mne.make_forward_solution(
        raw.info,
        trans=trans,
        src=src,
        bem=bem,
        meg=True,
        eeg=False,
        mindist=5.0,
        n_jobs=None,
        verbose=True,
    )
    

    forward_sol = f"{forward_sol_folder}/{sub_name}_SPONT_2_ico{src_num}_raw-fwd.fif"

   
    mne.write_forward_solution(forward_sol, fwd, overwrite=True, verbose=None)

Opening raw data file /mnt/isilon/w_neuro/gopalarlab/PASP_MEG_data/Resting_State_only/PASP003_kb_NR_AL/SPONT_2_kb_raw_quat_tsss.fif...


/tmp/ipykernel_729311/3523366605.py:35: RuntimeWarning: This filename (/mnt/isilon/w_neuro/gopalarlab/PASP_MEG_data/Resting_State_only/PASP003_kb_NR_AL/SPONT_2_kb_raw_quat_tsss.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw = mne.io.read_raw_fif(file_name)


    Range : 33000 ... 933999 =      6.600 ...   186.800 secs
Ready.
0 files missing from root.txt in /home/malann/mne_data/MNE-fsaverage-data
0 files missing from bem.txt in /home/malann/mne_data/MNE-fsaverage-data/fsaverage
Setting up the source space with the following parameters:

SUBJECTS_DIR = /home/malann/mne_data/MNE-fsaverage-data
Subject      = fsaverage
Surface      = white
Icosahedron subdivision grade 4

>>> 1. Creating the source space...

Doing the icosahedral vertex picking...
Loading /home/malann/mne_data/MNE-fsaverage-data/fsaverage/surf/lh.white...
Mapping lh fsaverage -> ico (4) ...
    Triangle neighbors and vertex normals...
Loading geometry from /home/malann/mne_data/MNE-fsaverage-data/fsaverage/surf/lh.sphere...
Setting up the triangulation for the decimated surface...
loaded lh.white 2562/163842 selected to source space (ico = 4)

Loading /home/malann/mne_data/MNE-fsaverage-data/fsaverage/surf/rh.white...
Mapping rh fsaverage -> ico (4) ...
    Triangle neighbor